# ALL Detection - Google Colab Training Quickstart

This notebook provides a 1-click environment to train the ALL Detection model using Google Colab's free GPUs, syncing directly with your GitHub repository and Google Drive.

## 1. Mount Google Drive
This gives Colab access to your dataset and a place to permanently save the trained models.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone Repository & Install Dependencies
Pulls the latest code from GitHub and installs PyTorch, Kornia, Albumentations, etc.

In [ ]:
%cd /content
!rm -rf ALL-Detection
!git clone https://github.com/Rujul-Rumale/ALL-Detection.git
%cd /content/ALL-Detection
!pip install -r requirements.txt

# Colab sometimes requires this to fix OpenCV headless issues:
!pip install opencv-python-headless


## 3. Link the Dataset
**Action Required**: You need your dataset in Google Drive.

It is fastest to upload your `C-NMC_Dataset` as a `.zip` file into Google Drive, copy it to the local Colab disk, and unzip it here. 
Update the `DRIVE_ZIP_PATH` below to match exactly where you uploaded the zip file in your Drive.

In [ ]:
# CHANGE THIS TO MATCH YOUR GOOGLE DRIVE PATH!
DRIVE_ZIP_PATH = "/content/drive/MyDrive/C-NMC_Dataset.zip"

import os
if not os.path.exists(DRIVE_ZIP_PATH):
    print(f"❌ ERROR: Dataset not found at {DRIVE_ZIP_PATH}")
    print("1. Ensure you have mounted Google Drive (Step 1)")
    print("2. Check if the filename in your Drive is exactly 'C-NMC_Dataset.zip'")
else:
    print(f"✅ Found dataset. Copying to local disk...")
    # Copy to /content/ for speed
    !cp "{DRIVE_ZIP_PATH}" /content/C-NMC_Dataset.zip
    print("📦 Unzipping (this may take 1-2 minutes)...")
    !unzip -q /content/C-NMC_Dataset.zip -d /content/ALL-Detection/
    !rm /content/C-NMC_Dataset.zip
    print("✨ Dataset ready in /content/ALL-Detection/")

## 4. Rebuild Data Splits for Linux Paths
The canonical cross-validation splits were built on Windows. This regenerates the `cv_splits_3fold.json` natively so all paths match `/content/ALL-Detection/...`.

In [ ]:
%cd /content/ALL-Detection
!python training_scripts/build_cv_splits.py

## 5. Begin Training!
Run the canonical training engine. We output the final `.pth` checkpoints and tensorboard logs directly back to your Google Drive so they aren't lost when Colab disconnects.

In [ ]:
%cd /content/ALL-Detection

# Ensure output directory exists on your Google Drive
!mkdir -p "/content/drive/MyDrive/ALL_Colab_Outputs"

# Train using high-performance Kornia engine (GPU augmentations)
# This usually speeds up training by 3-4x compared to CPU augmentations.
for fold in [1, 2, 3]:
    print(f"\n>>> Starting Training for Fold {fold}...")
    !python training_scripts/train_colab.py \
      --model mnv3l \
      --fold {fold} \
      --run_name mnv3l_kornia_fold{fold} \
      --batch_size 256 \
      --num_workers 4 \
      --no_live \
      --output_root "/content/drive/MyDrive/ALL_Colab_Outputs"